# 07 Explainability: SHAP on the trained classifiers

**Purpose:** explain which learning-activity features drive each XGBoost model's predictions, and compare with the paper's finding that `ncontent` and `nevents` are the most influential features.

**Models explained** (all trained earlier, loaded from `artifacts/models/`, nothing is retrained):
- `real_train_reference`: trained on real train (reference only)
- `ctgan_random_target`, `tvae_random_target`: trained on unbalanced synthetic data
- `ctgan_random_target_balanced`, `tvae_random_target_balanced`: trained on balanced synthetic data

**Data explained:** the real `test.csv` features (5,333 rows) with genuine NaNs preserved. SHAP only reads the data. **No model is fitted, tuned or selected here**, so using the test features does not leak anything into training. Features are in the fixed order `nevents, ndays_act, ncontent, nforum_posts`. The label is not passed to SHAP.

**Method:** `shap.TreeExplainer` on each XGBoost model. SHAP values are in **log-odds of class 1** (high engagement). Positive values push the prediction towards high engagement.

**Verification built in:** for each model, base value + sum of SHAP values must equal the model's raw output (`output_margin=True`) to numerical tolerance. A failure raises an error.

**Outputs**
- Global importance (mean |SHAP|) per model: `reports/shap_global_importance.json`, plus bar charts in `reports/figures/`
- Beeswarm summary for each model: `reports/figures/shap_summary_<model>.png`
- Local explanations (waterfall) for three real test students, for the `tvae_random_target` model: one correctly flagged high-engagement, one missed high-engagement (false negative), one false alarm (false positive)

**Interpretation limits (for the report)**
- SHAP describes **how the model behaves**, not what causes engagement. It does not show causal effects.
- A missing `nforum_posts` is not evidence of low engagement. Where the model uses it, that reflects the synthetic and real data patterns only.
- Local explanations are illustrations for three students. They are not guidance for decisions about any individual.

In [2]:
%pip install matplotlib

  Using cached matplotlib-3.11.2-cp311-cp311-win_amd64.whl.metadata (80 kB)
  Using cached contourpy-1.3.3-cp311-cp311-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.67.0-cp311-cp311-win_amd64.whl.metadata (139 kB)
  Using cached kiwisolver-1.5.1-cp311-cp311-win_amd64.whl.metadata (5.2 kB)
  Using cached pyparsing-3.3.3-py3-none-any.whl.metadata (5.9 kB)
Using cached matplotlib-3.11.2-cp311-cp311-win_amd64.whl (9.3 MB)
Using cached contourpy-1.3.3-cp311-cp311-win_amd64.whl (225 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
Using cached fonttools-4.67.0-cp311-cp311-win_amd64.whl (1.6 MB)
Using cached kiwisolver-1.5.1-cp311-cp311-win_amd64.whl (70 kB)
Using cached pyparsing-3.3.3-py3-none-any.whl (126 kB)

   ---------------------------------------- 0/6 [pyparsing]
   ------------- -------------------------- 2/6 [fonttools]
   ------------- -------------------------- 2/6 [fonttools]
   ------------- --

In [3]:
import json
from pathlib import Path

import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

FEATURES = ["nevents", "ndays_act", "ncontent", "nforum_posts"]
TARGET = "Engagement_Level"
SEED = 42
MODELS = [
    "real_train_reference",
    "ctgan_random_target",
    "tvae_random_target",
    "ctgan_random_target_balanced",
    "tvae_random_target_balanced",
]
LOCAL_MODEL = "tvae_random_target"


def find_project_root(start=None):
    """Walk upwards until a folder containing configs/base.yaml is found."""
    start = Path(start or Path.cwd()).resolve()
    for folder in [start, *start.parents]:
        if (folder / "configs" / "base.yaml").exists():
            return folder
    raise FileNotFoundError(f"Could not find configs/base.yaml above {start}")


def shap_for_model(model, X: pd.DataFrame):
    """Return (explainer, shap_values array of shape [n, 4]) for a binary XGBoost model."""
    explainer = shap.TreeExplainer(model)
    values = explainer.shap_values(X)
    if isinstance(values, list):          # some versions return one array per class
        values = values[-1]
    return explainer, np.asarray(values)


def check_additivity(model, explainer, values, X: pd.DataFrame, tol: float = 1e-3) -> float:
    """Base value + sum of SHAP values must equal the model's raw margin."""
    base = float(np.ravel(explainer.expected_value)[-1])
    margin = model.predict(X, output_margin=True)
    gap = float(np.max(np.abs(base + values.sum(axis=1) - margin)))
    if gap > tol:
        raise AssertionError(f"SHAP additivity check failed: max gap {gap:.5f}")
    return gap


PROJECT_ROOT = find_project_root()
model_dir = PROJECT_ROOT / "artifacts" / "models"
report_dir = PROJECT_ROOT / "reports"
fig_dir = report_dir / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)
matplotlib.use("Agg")

test = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "test.csv")[FEATURES + [TARGET]]
X_test, y_test = test[FEATURES], test[TARGET].astype(int)
print("Test:", test.shape, "| NaN % per feature:", (X_test.isna().mean() * 100).round(2).to_dict())

importance, store = {}, {}
for name in MODELS:
    path = model_dir / f"{name}.joblib"
    if not path.exists():
        print(f"{name}: model file not found, skipped")
        continue
    model = joblib.load(path)
    explainer, values = shap_for_model(model, X_test)
    gap = check_additivity(model, explainer, values, X_test)
    store[name] = (model, explainer, values)

    mean_abs = pd.Series(np.abs(values).mean(axis=0), index=FEATURES)
    importance[name] = mean_abs.round(4).to_dict()
    print(f"{name}: additivity gap {gap:.2e} | ranking: {list(mean_abs.sort_values(ascending=False).index)}")

    fig, ax = plt.subplots(figsize=(5, 3))
    mean_abs.sort_values().plot.barh(ax=ax, color="#4c72b0")
    ax.set_xlabel("mean |SHAP| (log-odds of high engagement)")
    ax.set_title(name)
    fig.tight_layout()
    fig.savefig(fig_dir / f"shap_importance_{name}.png", dpi=150)
    plt.close(fig)

    shap.summary_plot(values, X_test, feature_names=FEATURES, show=False)
    plt.tight_layout()
    plt.savefig(fig_dir / f"shap_summary_{name}.png", dpi=150)
    plt.close()

print("\nMean |SHAP| per model")
print(pd.DataFrame(importance).round(3))

# Local explanations for one model: TP, FN, FP on the real test set
if LOCAL_MODEL in store:
    model, explainer, values = store[LOCAL_MODEL]
    pred = model.predict(X_test)
    groups = {
        "true_positive": np.where((y_test == 1) & (pred == 1))[0],
        "false_negative": np.where((y_test == 1) & (pred == 0))[0],
        "false_positive": np.where((y_test == 0) & (pred == 1))[0],
    }
    rng = np.random.default_rng(SEED)
    base = float(np.ravel(explainer.expected_value)[-1])
    chosen = {}
    for label, idx in groups.items():
        i = int(rng.choice(idx))
        chosen[label] = i
        exp = shap.Explanation(values=values[i], base_values=base,
                               data=X_test.iloc[i].values, feature_names=FEATURES)
        shap.plots.waterfall(exp, show=False)
        plt.tight_layout()
        plt.savefig(fig_dir / f"shap_local_{LOCAL_MODEL}_{label}.png", dpi=150, bbox_inches="tight")
        plt.close()
        print(f"\n{label} (test row {i}, true={int(y_test.iloc[i])}, "
              f"P(high)={float(model.predict_proba(X_test.iloc[[i]])[0, 1]):.3f})")
        print(pd.DataFrame({"value": X_test.iloc[i], "shap": values[i].round(3)}).to_string())

out = {
    "explained_data": "real test features (SHAP only reads the data; nothing is fitted or tuned)",
    "units": "mean |SHAP| in log-odds of class 1",
    "mean_abs_shap": importance,
    "local_model": LOCAL_MODEL,
    "local_rows": chosen if LOCAL_MODEL in store else None,
}
(report_dir / "shap_global_importance.json").write_text(json.dumps(out, indent=2), encoding="utf-8")
print("\nSaved:", report_dir / "shap_global_importance.json")

Test: (5333, 5) | NaN % per feature: {'nevents': 17.1, 'ndays_act': 10.09, 'ncontent': 0.0, 'nforum_posts': 67.84}
real_train_reference: additivity gap 9.54e-06 | ranking: ['ncontent', 'nevents', 'nforum_posts', 'ndays_act']
ctgan_random_target: additivity gap 4.77e-06 | ranking: ['ncontent', 'nevents', 'ndays_act', 'nforum_posts']
tvae_random_target: additivity gap 9.54e-06 | ranking: ['ncontent', 'nevents', 'nforum_posts', 'ndays_act']
ctgan_random_target_balanced: additivity gap 4.29e-06 | ranking: ['ncontent', 'nevents', 'ndays_act', 'nforum_posts']
tvae_random_target_balanced: additivity gap 1.05e-05 | ranking: ['ncontent', 'nevents', 'ndays_act', 'nforum_posts']

Mean |SHAP| per model
              real_train_reference  ctgan_random_target  tvae_random_target  \
nevents                      1.008                0.726               1.298   
ndays_act                    0.231                0.343               0.191   
ncontent                     3.476                1.713        

## Results: SHAP global importance (real test features, five models)

**Setup:** `shap.TreeExplainer` on five trained XGBoost models loaded from `artifacts/models/`. Explained data: real `test.csv` features (5,333 rows, genuine NaNs). Nothing was fitted or tuned. Additivity check (base value + sum of SHAP = raw margin) passed for every model, with a maximum gap of 1.05e-05. Saved in `reports/shap_global_importance.json`; plots in `reports/figures/`.

**Measured: mean |SHAP| (log-odds of high engagement)**

| Feature | Real-train ref. | CTGAN | TVAE | CTGAN balanced | TVAE balanced |
|---|---|---|---|---|---|
| ncontent | 3.476 | 1.713 | 3.431 | 1.683 | 3.859 |
| nevents | 1.008 | 0.726 | 1.298 | 0.710 | 1.153 |
| nforum_posts | 0.267 | 0.244 | 0.308 | 0.238 | 0.246 |
| ndays_act | 0.231 | 0.343 | 0.191 | 0.310 | 0.315 |

**Interpretation**
- `ncontent` ranks first and `nevents` second in all five models, including the real-data reference. This matches the paper's reported finding that `ncontent` and `nevents` are the most influential features.
- Ranks 3 and 4 (`ndays_act`, `nforum_posts`) are close in magnitude and swap between models, so their order is not interpreted.
- As a share of total mean |SHAP|, the TVAE models are close to the real-data model (`ncontent` 66 to 70%, `nevents` 21 to 25%), while the CTGAN models spread more weight onto `ndays_act` and `nforum_posts` (`ncontent` 57%). One seed, observation only.
- Mean |SHAP| is on each model's own log-odds scale, so shares, not raw values, are compared across models.
- `nforum_posts` carries 4 to 8% of total importance. Whether missingness of `nforum_posts` itself is used by the models is **not tested** here, because mean |SHAP| mixes observed and missing rows.

**Interpretation limits:** SHAP describes model behaviour, not causes of engagement. Local explanations are illustrations for single test rows and are not guidance for decisions about individual students.

**Limitations:** one seed per generator; SHAP computed on the test features only.